# 4 · Reranking ablation

After fusion, `retrieve()` can rerank its candidate pool with a cross-encoder (`FV_RERANK=true`,
fastembed `TextCrossEncoder`). It also computes a cheap lexical containment score for every candidate,
which drives abstention. This notebook asks whether reranking pays for its latency:

* **none**: RRF order, as served by default;
* **lexical**: candidates re-sorted by query containment (share of the query's content words in the chunk);
* **rrf+lexical**: reciprocal rank fusion of the RRF order and the lexical order;
* **cross-encoder**: the served reranker model (full mode only, when the model loads).

Two tasks: the catalog label queries (document-level, graded) and `qa-small` over the Atlas brief mixed
into catalog distractors (a chunk is relevant when it contains the evidence span). The rerank depth `N`
(how many fused candidates are re-scored) is swept as well.

In [1]:
import os
import sys

sys.path.insert(0, os.environ.get("FV_NOTEBOOK_DIR", os.getcwd()))
import fvnb

fvnb.setup()
print("smoke mode" if fvnb.SMOKE else "full mode")

full mode


## Index and candidate pools

In [2]:
from collections import Counter

from folioverse_ai.retrieval.retrieve import retrieve

docs = fvnb.catalog()[: fvnb.sample(400, 60)]


async def build():
    svids = [await fvnb.ingest(f"# {d['title']}\n\n{d['description']}", d["title"]) for d in docs]
    pdf = await fvnb.ingest(
        (fvnb.DATA / "synthetic/docs/atlas-brief.pdf").read_bytes(), "atlas-brief.pdf", "application/pdf"
    )
    return svids, pdf


svids, atlas = fvnb.arun(build())
doc_of = {s: i for i, s in enumerate(svids)}

pairs = Counter((g, th["name"]) for d in docs for g in d["genres"] for th in d["themes"])
label_queries = []
for (genre, theme), n in sorted(pairs.items()):
    if n >= 3:
        gains = {
            i: max(th["weight"] for th in d["themes"] if th["name"] == theme)
            for i, d in enumerate(docs)
            if genre in d["genres"] and any(th["name"] == theme for th in d["themes"])
        }
        label_queries.append({"text": f"{genre.lower()} stories about {theme}", "gains": gains})
label_queries = label_queries[: fvnb.sample(100, 12)]
qa = [q for q in fvnb.qa_small() if q["answerable"]]

POOL_TOPK = 10  # retrieve() pool = max(4 * topK, 20) = 40 candidates


async def pools(texts: list[str], scope):
    out = []
    for t in texts:
        r = await retrieve(t, scope, {"topK": POOL_TOPK, "rerank": False})
        rows = await fvnb.repo().get_chunks([c.chunk_id for c in r.candidates])
        out.append([(c, rows[c.chunk_id]["text"]) for c in r.candidates if c.chunk_id in rows])
    return out


label_pools = fvnb.arun(pools([q["text"] for q in label_queries], fvnb.scope(svids)))
qa_pools = fvnb.arun(pools([q["question"] for q in qa], fvnb.scope([*svids, atlas])))
print(
    f"{len(label_queries)} label queries, {len(qa)} answerable qa-small questions, pool {len(label_pools[0])}"
)

100 label queries, 10 answerable qa-small questions, pool 40


## Rerankers

In [3]:
import time

from folioverse_ai.text import containment


def by_lexical(query, pool):
    return sorted(pool, key=lambda ct: -containment(query, ct[1]))  # stable: ties keep RRF order


def by_rrf_lexical(query, pool, k=60):
    lex = {id(ct): r for r, ct in enumerate(by_lexical(query, pool))}
    return sorted(pool, key=lambda ct: -(1 / (k + pool.index(ct)) + 1 / (k + lex[id(ct)])))


rerankers = {"none": lambda q, pool: list(pool), "lexical": by_lexical, "rrf+lexical": by_rrf_lexical}
cross_model = None
if not fvnb.SMOKE:
    try:
        from folioverse_ai.retrieval.embedder import Reranker
        from folioverse_ai.settings import get_settings

        cross_model = get_settings().rerank_model
        ce = Reranker(cross_model)

        def by_cross(query, pool):
            scores = ce.score(query, [t for _, t in pool])
            order = sorted(range(len(pool)), key=lambda i: -scores[i])
            return [pool[i] for i in order]

        rerankers["cross-encoder"] = by_cross
    except Exception as e:
        cross_model = None
        print("cross-encoder unavailable:", e.__class__.__name__)
print("rerankers:", ", ".join(rerankers))

rerankers: none, lexical, rrf+lexical, cross-encoder


## Ablation at full depth

In [4]:


def rerank_depth(fn, query, pool, depth):
    head = fn(query, pool[:depth]) if depth else []
    return head + pool[depth:]


def evaluate(name, depth):
    lat, nd, r5, mrr_l = [], [], [], []
    for q, pool in zip(label_queries, label_pools, strict=True):
        t0 = time.perf_counter()
        ranked = rerank_depth(rerankers[name], q["text"], pool, depth)
        lat.append((time.perf_counter() - t0) * 1000)
        ids = [doc_of.get(c.source_version_id) for c, _ in ranked]
        nd.append(fvnb.ndcg(ids, q["gains"], 10))
        r5.append(fvnb.recall(ids, set(q["gains"]), 5))
        mrr_l.append(fvnb.mrr(ids, set(q["gains"])))
    hit1, hit3, hit5, mrr_q = [], [], [], []
    for q, pool in zip(qa, qa_pools, strict=True):
        t0 = time.perf_counter()
        ranked = rerank_depth(rerankers[name], q["question"], pool, depth)
        lat.append((time.perf_counter() - t0) * 1000)
        rel = [q["evidence"].lower() in t.lower() for _, t in ranked]
        hit1.append(float(rel[0]) if rel else 0.0)
        hit3.append(float(any(rel[:3])))
        hit5.append(float(any(rel[:5])))
        mrr_q.append(next((1 / (i + 1) for i, x in enumerate(rel) if x), 0.0))
    return {
        "reranker": name,
        "depth": depth,
        "label_ndcg@10": fvnb.mean(nd),
        "label_recall@5": fvnb.mean(r5),
        "label_mrr": fvnb.mean(mrr_l),
        "qa_hit@1": fvnb.mean(hit1),
        "qa_hit@3": fvnb.mean(hit3),
        "qa_hit@5": fvnb.mean(hit5),
        "qa_mrr": fvnb.mean(mrr_q),
        "rerank_ms_p50": fvnb.percentile(lat, 0.5),
    }


FULL = len(label_pools[0])
ablation = [evaluate(name, FULL) for name in rerankers]
fvnb.table(ablation)

| reranker | depth | label_ndcg@10 | label_recall@5 | label_mrr | qa_hit@1 | qa_hit@3 | qa_hit@5 | qa_mrr | rerank_ms_p50 |
|---|---|---|---|---|---|---|---|---|---|
| none | 40 | 0.186 | 0.101 | 0.324 | 1.000 | 1.000 | 1.000 | 1.000 | 0.002 |
| lexical | 40 | 0.215 | 0.108 | 0.358 | 1.000 | 1.000 | 1.000 | 1.000 | 1.631 |
| rrf+lexical | 40 | 0.203 | 0.104 | 0.338 | 1.000 | 1.000 | 1.000 | 1.000 | 2.863 |
| cross-encoder | 40 | 0.379 | 0.230 | 0.670 | 1.000 | 1.000 | 1.000 | 1.000 | 699.654 |

## Rerank depth
Only the top `N` fused candidates are re-scored; the rest keep their RRF order.

In [5]:
depths = [5, 10, 20, FULL]
depth_rows = [evaluate(name, n) for name in rerankers if name != "none" for n in depths]
fvnb.table(depth_rows, ["reranker", "depth", "label_ndcg@10", "qa_hit@1", "qa_mrr", "rerank_ms_p50"])

| reranker | depth | label_ndcg@10 | qa_hit@1 | qa_mrr | rerank_ms_p50 |
|---|---|---|---|---|---|
| lexical | 5 | 0.192 | 1.000 | 1.000 | 0.284 |
| lexical | 10 | 0.194 | 1.000 | 1.000 | 0.554 |
| lexical | 20 | 0.212 | 1.000 | 1.000 | 0.919 |
| lexical | 40 | 0.215 | 1.000 | 1.000 | 1.767 |
| rrf+lexical | 5 | 0.187 | 1.000 | 1.000 | 0.263 |
| rrf+lexical | 10 | 0.190 | 1.000 | 1.000 | 0.514 |
| rrf+lexical | 20 | 0.197 | 1.000 | 1.000 | 1.145 |
| rrf+lexical | 40 | 0.203 | 1.000 | 1.000 | 3.325 |
| cross-encoder | 5 | 0.214 | 1.000 | 1.000 | 76.747 |
| cross-encoder | 10 | 0.234 | 1.000 | 1.000 | 140.248 |
| cross-encoder | 20 | 0.314 | 1.000 | 1.000 | 302.546 |
| cross-encoder | 40 | 0.379 | 1.000 | 1.000 | 675.136 |

In [6]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2.8))
for name in [n for n in rerankers if n != "none"]:
    rows = [r for r in depth_rows if r["reranker"] == name]
    ax.plot([r["depth"] for r in rows], [r["qa_mrr"] for r in rows], marker="o", label=name)
ax.axhline(ablation[0]["qa_mrr"], color="grey", linestyle="--", label="none")
ax.set_xlabel("rerank depth N")
ax.set_ylabel("qa-small MRR")
ax.legend(fontsize=8)
fig.tight_layout()

## Findings

In [7]:
base = ablation[0]
for r in ablation[1:]:
    fvnb.finding(
        f"{r['reranker']}: label nDCG@10 {r['label_ndcg@10'] - base['label_ndcg@10']:+.3f}, "
        f"qa hit@1 {r['qa_hit@1'] - base['qa_hit@1']:+.3f}, qa MRR {r['qa_mrr'] - base['qa_mrr']:+.3f} vs RRF only, "
        f"at {r['rerank_ms_p50']:.2f} ms p50 per query."
    )
if base["qa_hit@3"] == 1.0:
    fvnb.finding(
        "qa-small is saturated at hit@3 for RRF alone; its questions share vocabulary with the evidence."
    )
if cross_model is None:
    fvnb.finding("Cross-encoder not evaluated in this run (smoke mode or model unavailable offline).")

- lexical: label nDCG@10 +0.029, qa hit@1 +0.000, qa MRR +0.000 vs RRF only, at 1.63 ms p50 per query.
- rrf+lexical: label nDCG@10 +0.016, qa hit@1 +0.000, qa MRR +0.000 vs RRF only, at 2.86 ms p50 per query.
- cross-encoder: label nDCG@10 +0.193, qa hit@1 +0.000, qa MRR +0.000 vs RRF only, at 699.65 ms p50 per query.
- qa-small is saturated at hit@3 for RRF alone; its questions share vocabulary with the evidence.


In [8]:
fvnb.save(
    "04-reranking-ablation",
    {
        "documents": len(docs),
        "labelQueries": len(label_queries),
        "qaQuestions": len(qa),
        "pool": FULL,
        "crossEncoder": cross_model,
        "ablation": ablation,
        "depthSweep": depth_rows,
    },
)

saved datasets/evaluation/results/nb-04-reranking-ablation.json
